# 🏥 ATM-Net++: Lumbar Spine MRI Segmentation
## Kaggle Version — P100 16GB GPU

### ✅ Advantages over Google Colab:
- **No disconnection** — runs for 9 hours unattended
- **P100 16GB VRAM** (same as Colab T4 but more stable)
- **30 hours/week free** GPU quota
- Results saved to `/kaggle/working/` — downloadable after
- Can run fully in background — close browser, training continues

### 📋 Setup Steps:
1. Upload SPIDER dataset as a Kaggle Dataset (see Step 1)
2. Add it to this notebook
3. Enable GPU: Settings → Accelerator → **GPU P100**
4. Click **Run All**

### ⏱️ Expected time:
- ~45 sec/epoch on P100
- 200 epochs = ~2.5 hours
- **Dice >0.85 in ~2 hours**
- **Dice >0.90 in ~3 hours**

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 1: Check environment
# ═══════════════════════════════════════════════════════
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
else:
    print('WARNING: No GPU — go to Settings > Accelerator > GPU P100')

# Check if dataset is attached
print()
import os
for path in ['/kaggle/input/', '/kaggle/working/']:
    if os.path.exists(path):
        contents = os.listdir(path)
        print(f'{path}: {contents[:5]}')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 2: Install correct PyTorch + dependencies
# P100 = sm_60, needs PyTorch with CUDA 11.8 support
# ═══════════════════════════════════════════════════════
import torch, subprocess

# Check if current PyTorch works with this GPU
gpu_works = torch.cuda.is_available()
if torch.cuda.is_available():
    try:
        x = torch.randn(3,3).cuda()
        del x; torch.cuda.empty_cache()
        print('✓ Current PyTorch works with GPU — skipping reinstall')
    except Exception as e:
        gpu_works = False
        print(f'GPU test failed: {e}')

if not gpu_works:
    print('Installing PyTorch 2.1 (compatible with P100 sm_60)...')
    import subprocess
    subprocess.run(['pip','uninstall','torch','torchvision','-y','-q'], capture_output=True)
    result = subprocess.run(
        ['pip','install','torch==2.1.0','torchvision==0.16.0',
         '--index-url','https://download.pytorch.org/whl/cu118','-q'],
        capture_output=True, text=True
    )
    print(result.stdout[-200:] if result.stdout else 'done')
    print('PyTorch 2.1 installed — please RESTART the kernel and re-run from Cell 1')
    print('Kernel > Restart & Run All')

# Install other dependencies
!pip install SimpleITK opencv-python-headless scipy pandas -q

# Final check
import importlib, torch as t2
print(f'\nPyTorch : {t2.__version__}')
print(f'CUDA    : {t2.cuda.is_available()}')
if t2.cuda.is_available():
    test = t2.randn(100,100).cuda()
    print(f'GPU OK  : {test.device}')
    del test
print('✓ Ready')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 3: Auto-detect dataset location
# ═══════════════════════════════════════════════════════
import glob, os, shutil
from pathlib import Path

# Search for dataset in all Kaggle input locations
DATA_ROOT = None
search_locations = [
    '/kaggle/input/spider-lumbar-spine-mri/10159290',
    '/kaggle/input/spider-lumbar-spine/10159290',
    '/kaggle/input/spine-segmentation/10159290',
    '/kaggle/input/',
]

# Auto-find by looking for overview.csv
all_csvs = glob.glob('/kaggle/input/**/overview.csv', recursive=True)
if all_csvs:
    DATA_ROOT = str(Path(all_csvs[0]).parent)
    print(f'✓ Found dataset at: {DATA_ROOT}')
else:
    # Try to find images
    all_mha = glob.glob('/kaggle/input/**/*.mha', recursive=True)
    if all_mha:
        DATA_ROOT = str(Path(all_mha[0]).parent.parent)
        print(f'✓ Found MHA files, guessing DATA_ROOT: {DATA_ROOT}')
    else:
        print('⚠ Dataset not found in /kaggle/input/')
        print()
        print('TO ADD DATASET:')
        print('1. Go to https://www.kaggle.com/datasets/new')
        print('2. Upload your SPIDER folder (images/, masks/, overview.csv)')
        print('3. In this notebook: + Add Data → Your Datasets → Select it')
        print('4. Re-run this cell')
        print()
        print('Available datasets:', os.listdir('/kaggle/input/'))

if DATA_ROOT:
    IMAGES_DIR = Path(DATA_ROOT) / 'images'
    MASKS_DIR  = Path(DATA_ROOT) / 'masks'
    OVERVIEW   = Path(DATA_ROOT) / 'overview.csv'
    n_img = len(list(IMAGES_DIR.glob('*.mha'))) if IMAGES_DIR.exists() else 0
    n_msk = len(list(MASKS_DIR.glob('*.mha')))  if MASKS_DIR.exists()  else 0
    print(f'  Images : {n_img}')
    print(f'  Masks  : {n_msk}')
    print(f'  CSV    : {OVERVIEW.exists()}')
    if n_img > 400: print('✓ Dataset ready!')
    else:
        print('⚠ Images not found at expected path. Searching...')
        all_t2 = glob.glob('/kaggle/input/**/*_t2.mha', recursive=True)
        if all_t2:
            DATA_ROOT = str(Path(all_t2[0]).parent.parent)
            IMAGES_DIR = Path(all_t2[0]).parent
            print(f'  Found images at: {IMAGES_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 4: FULL TRAINING — Runs all 200 epochs
# ~45 sec/epoch on P100 = ~2.5 hours total
# Results saved to /kaggle/working/best_model.pth
# ═══════════════════════════════════════════════════════
import sys, os, time, warnings, json, random, gc
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, cv2
import SimpleITK as sitk
from pathlib import Path
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── Config ────────────────────────────────────────────
assert DATA_ROOT is not None, 'Run Step 3 first to detect dataset'

IMAGES_DIR  = Path(DATA_ROOT) / 'images'
MASKS_DIR   = Path(DATA_ROOT) / 'masks'
OVERVIEW    = Path(DATA_ROOT) / 'overview.csv'
OUTPUT_DIR  = Path('/kaggle/working')
CKPT_BEST   = OUTPUT_DIR / 'best_model.pth'
CACHE_DIR   = OUTPUT_DIR / 'cache'; CACHE_DIR.mkdir(exist_ok=True)

# Filter T2-only for speed (T1 files would double epoch time)
T2_ONLY   = True   # Set False to use both T1+T2
IMG_SIZE  = 384    # 384x384 — optimal for P100 16GB
BATCH_SIZE= 8      # 8 x 384x384 fp16 = fits P100 16GB
EPOCHS    = 200
LR        = 2e-4
WD        = 5e-4
MAX_SPP   = 30     # slices per patient
NC        = 19
SEED      = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

S2A={**{i:i for i in range(1,9)},100:9,**{201+i:10+i for i in range(8)}}
CN={0:'bg',1:'Vert-1',2:'Vert-2',3:'Vert-3',4:'Vert-4',5:'Vert-5',6:'Vert-6',
    7:'Vert-7',8:'Vert-8',9:'Sacrum',10:'IVD-1',11:'IVD-2',12:'IVD-3',
    13:'IVD-4',14:'IVD-5',15:'IVD-6',16:'IVD-7',17:'IVD-8',18:'Canal'}
RARE=[7,8,16,17]
CW=torch.tensor([0,1,1,1,1.5,2,3,6,12,1,6,4,4,5,6,9,14,30,0]).float()

def remap(m):
    o=np.zeros_like(m,dtype=np.int32)
    for s,d in S2A.items(): o[m==s]=d
    return o

def fg(m): return float((m>0).sum())/max(m.size,1)

# ── Data ──────────────────────────────────────────────
def build_cache(pids, split):
    suffix = 't2' if T2_ONLY else 'all'
    cf = CACHE_DIR/f'{split}_{suffix}_{IMG_SIZE}.npz'
    if cf.exists():
        d=np.load(cf); n=len(d['imgs'])
        print(f'  {split}: {n} slices (from cache)'); return d['imgs'],d['msks'],d['rare'].tolist()
    print(f'  {split}: processing {len(pids)} patients...')
    imgs,msks,rare=[],[],[]; skipped=0
    modalities = ['t2'] if T2_ONLY else ['t1','t2']
    for i,pid in enumerate(pids):
        for mod in modalities:
            ip=IMAGES_DIR/f'{pid}_{mod}.mha'; mp=MASKS_DIR/f'{pid}_{mod}.mha'
            if not ip.exists() or not mp.exists(): skipped+=1; continue
            try:
                iv=sitk.GetArrayFromImage(sitk.ReadImage(str(ip))).astype(np.float32)
                mv=sitk.GetArrayFromImage(sitk.ReadImage(str(mp))).astype(np.int32)
            except: skipped+=1; continue
            n=iv.shape[0]; lo,hi=int(n*0.05),int(n*0.95)
            ranked=sorted(range(lo,hi),key=lambda s:fg(remap(mv[s])),reverse=True)[:MAX_SPP]
            for s in ranked:
                rm=remap(mv[s])
                if fg(rm)<0.005: continue
                p1,p99=np.percentile(iv[s],[0.5,99.5])
                ir=cv2.resize(np.clip((iv[s]-p1)/(p99-p1+1e-8),0,1).astype(np.float32),
                              (IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR).astype(np.float16)
                mr=cv2.resize(rm.astype(np.float32),(IMG_SIZE,IMG_SIZE),
                              interpolation=cv2.INTER_NEAREST).astype(np.uint8)
                imgs.append(ir); msks.append(np.clip(mr,0,NC-1))
                rare.append(1.0 if float(sum((rm==c).sum() for c in RARE))/max(rm.size,1)>0.0003 else 0.1)
        if (i+1)%30==0: print(f'    {i+1}/{len(pids)}, {len(imgs)} slices...')
    if not imgs: raise ValueError(f'No slices! Check IMAGES_DIR={IMAGES_DIR} (exists={IMAGES_DIR.exists()})')
    print(f'  {split}: {len(imgs)} slices ({skipped} skipped)')
    np.savez_compressed(cf,imgs=np.stack(imgs).astype(np.float16),
                        msks=np.stack(msks).astype(np.uint8),rare=np.array(rare,dtype=np.float32))
    return np.stack(imgs).astype(np.float16),np.stack(msks).astype(np.uint8),rare

class Aug:
    def __call__(self,img,msk):
        if random.random()<0.5: img=np.fliplr(img).copy();msk=np.fliplr(msk).copy()
        if random.random()<0.6:
            M=cv2.getRotationMatrix2D((IMG_SIZE//2,IMG_SIZE//2),random.uniform(-20,20),1)
            img=cv2.warpAffine(img.astype(np.float32),M,(IMG_SIZE,IMG_SIZE),
                               flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
            mf=cv2.warpAffine(msk.astype(np.float32),M,(IMG_SIZE,IMG_SIZE),
                               flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            msk=np.clip(mf.astype(np.int32),0,NC-1)
        img=np.clip(np.power(img.astype(np.float32)+1e-8,random.uniform(0.65,1.5)),0,1)
        img=np.clip(img*random.uniform(0.75,1.25)+random.uniform(-0.1,0.1),0,1)
        return img.astype(np.float32),msk.astype(np.int64)

class DS(Dataset):
    def __init__(self,i,m,a=None): self.i=i;self.m=m;self.a=a
    def __len__(self): return len(self.i)
    def __getitem__(self,x):
        img=self.i[x].astype(np.float32);msk=self.m[x].astype(np.int64)
        if self.a: img,msk=self.a(img,msk)
        return torch.from_numpy(img[None]).float(),torch.from_numpy(msk).long()

# ── Model ─────────────────────────────────────────────
class CA(nn.Module):
    def __init__(self,ch,r=8):
        super().__init__();r=max(1,ch//r)
        self.a=nn.AdaptiveAvgPool2d(1);self.b=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(ch,r),nn.ReLU(True),nn.Linear(r,ch),nn.Sigmoid())
    def forward(self,x): return x*(self.fc(self.a(x))+self.fc(self.b(x))).clamp(0,1).view(x.shape[0],-1,1,1)

class SA(nn.Module):
    def __init__(self):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(2,1,7,padding=3,bias=False),nn.BatchNorm2d(1),nn.Sigmoid())
    def forward(self,x): return x*self.c(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1))

class RB(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.n=nn.Sequential(nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch),nn.ReLU(True),
                             nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch))
        self.ca=CA(ch);self.sa=SA();self.act=nn.ReLU(True)
    def forward(self,x): return self.act(self.sa(self.ca(self.n(x)))+x)

class Enc(nn.Module):
    def __init__(self,ci,co,d=0):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(ci,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True),
                             nn.Conv2d(co,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True))
        self.r=RB(co);self.d=nn.Dropout2d(d) if d>0 else nn.Identity()
    def forward(self,x): return self.d(self.r(self.c(x)))

class Net(nn.Module):
    def __init__(self,b=32,nc=NC,d=0.2):
        super().__init__()
        self.e1=Enc(1,b);self.e2=Enc(b,b*2,d*0.3);self.e3=Enc(b*2,b*4,d*0.6);self.e4=Enc(b*4,b*8,d*0.8)
        self.bn=nn.Sequential(Enc(b*8,b*16,d),nn.Dropout2d(d));self.p=nn.MaxPool2d(2)
        self.u4=nn.ConvTranspose2d(b*16,b*8,2,2);self.d4=Enc(b*16,b*8,d*0.4)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2);self.d3=Enc(b*8,b*4,d*0.2)
        self.u2=nn.ConvTranspose2d(b*4,b*2,2,2);self.d2=Enc(b*4,b*2)
        self.u1=nn.ConvTranspose2d(b*2,b,2,2);self.d1=Enc(b*2,b)
        self.s3=nn.Conv2d(b*4,nc,1);self.s2=nn.Conv2d(b*2,nc,1);self.o=nn.Conv2d(b,nc,1)
        self.ax=nn.Sequential(nn.Conv2d(b,b,3,1,1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,nc,1))
    def forward(self,x):
        sz=x.shape[2:]
        e1=self.e1(x);e2=self.e2(self.p(e1));e3=self.e3(self.p(e2));e4=self.e4(self.p(e3))
        d=self.bn(self.p(e4))
        d=self.d4(torch.cat([self.u4(d),e4],1));d=self.d3(torch.cat([self.u3(d),e3],1))
        o3=F.interpolate(self.s3(d),sz,mode='bilinear',align_corners=False)
        d=self.d2(torch.cat([self.u2(d),e2],1))
        o2=F.interpolate(self.s2(d),sz,mode='bilinear',align_corners=False)
        d=self.d1(torch.cat([self.u1(d),e1],1))
        return (self.o(d),o2,o3,self.ax(d)) if self.training else self.o(d)

# ── Losses ────────────────────────────────────────────
def dice_w(lg,tg,sm=1e-6):
    B,C,H,W=lg.shape;s=F.softmax(lg,1);o=F.one_hot(tg.clamp(0,C-1),C).permute(0,3,1,2).float()
    p=s[:,1:].reshape(B,C-1,-1);t=o[:,1:].reshape(B,C-1,-1)
    inter=(p*t).sum(-1);union=p.sum(-1)+t.sum(-1);mask=(t.sum(-1)>0).float()
    w=CW[1:].to(lg.device).view(1,C-1)
    return 1-((2*inter+sm)/(union+sm)*mask*w).sum()/(mask*w).sum().clamp(min=1)

def focal(lg,tg,g=2):
    ce=F.cross_entropy(lg,tg.clamp(0,NC-1),reduction='none'); return ((1-torch.exp(-ce))**g*ce).mean()

def boundary(lg,tg):
    s=F.softmax(lg,1);o=F.one_hot(tg.clamp(0,NC-1),NC).permute(0,3,1,2).float()
    b=(F.max_pool2d(o[:,1:],3,stride=1,padding=1)-o[:,1:]).clamp(0,1)
    w=CW[1:].to(lg.device).view(1,-1,1,1)
    return (b*(1-s[:,1:])*w).sum()/((b*w).sum()+1e-6)

def comp(lg,tg):
    tc=tg.clamp(0,NC-1)
    return F.cross_entropy(lg,tc,label_smoothing=0.03)+dice_w(lg,tc)+0.3*focal(lg,tc)+0.15*boundary(lg,tc)

def loss_fn(outs,tg):
    o1,o2,o3,ax=outs;main=comp(o1,tg)+0.3*comp(o2,tg)+0.15*comp(o3,tg)
    tc=tg.clamp(0,NC-1);rm=sum((tc==c).float() for c in RARE).clamp(0,1)
    return main+0.3*(F.cross_entropy(ax,tc,reduction='none')*(1+4*rm)).mean()

@torch.no_grad()
def get_dice(lg,tg):
    pred=lg.argmax(1).cpu().numpy();gt=tg.cpu().numpy();sm=1e-6;D=defaultdict(list)
    for b in range(pred.shape[0]):
        for c in range(1,NC):
            p=(pred[b]==c).astype(float).ravel();t=(gt[b]==c).astype(float).ravel()
            if t.sum()==0 and p.sum()==0: continue
            tp=(p*t).sum();fp=(p*(1-t)).sum();fn=((1-p)*t).sum()
            D[c].append((2*tp+sm)/(2*tp+fp+fn+sm))
    all_d=[v for vs in D.values() for v in vs]
    return D,float(np.mean(all_d)) if all_d else 0

def get_splits():
    df=pd.read_csv(OVERVIEW);tr,va=[],[]; seen=set()
    for n in df['new_file_name'].tolist():
        if not n.endswith('_t2') or 'SPACE' in n: continue
        p=n.replace('_t2','')
        if p in seen: continue
        seen.add(p);s=df.loc[df['new_file_name']==n,'subset'].values
        (va if len(s) and s[0]=='validation' else tr).append(p)
    return tr,va

# ── Main Training ─────────────────────────────────────
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('='*62)
print(f'  ATM-Net++ | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  {IMG_SIZE}x{IMG_SIZE} | BS={BATCH_SIZE} | T2_ONLY={T2_ONLY} | AMP=ON')
print('='*62)

tr_pids,va_pids=get_splits()
found=sum(1 for p in tr_pids if (IMAGES_DIR/f'{p}_t2.mha').exists())
print(f'\nPatients: {len(tr_pids)} train | {len(va_pids)} val | Files found: {found}\n')
assert found>0, f'No files found! Check IMAGES_DIR={IMAGES_DIR}'

print('Building data cache...')
ti,tm,tr_rare=build_cache(tr_pids,'train')
vi,vm,va_rare=build_cache(va_pids,'val')
print(f'RAM: ~{(ti.nbytes+tm.nbytes+vi.nbytes+vm.nbytes)//1024**2} MB\n')

sampler=WeightedRandomSampler(torch.tensor(tr_rare),len(tr_rare),replacement=True)
tr_dl=DataLoader(DS(ti,tm,Aug()),batch_size=BATCH_SIZE,sampler=sampler,num_workers=4,pin_memory=True)
va_dl=DataLoader(DS(vi,vm),batch_size=BATCH_SIZE,shuffle=False,num_workers=4,pin_memory=True)

model=Net(b=32,nc=NC,d=0.2).to(device)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params')
print(f'Batches: {len(tr_dl)} train | {len(va_dl)} val\n')

start_ep=1;best=0.0
if CKPT_BEST.exists():
    ck=torch.load(str(CKPT_BEST),map_location=device)
    model.load_state_dict(ck['model_state_dict'],strict=False)
    best=ck.get('best_dice',0);start_ep=ck.get('epoch',0)+1
    print(f'Resumed: epoch {ck.get("epoch")} | dice={best:.4f}\n')

optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(optim,T_max=EPOCHS,eta_min=1e-6)
scaler=GradScaler();no_imp=0;t0tot=time.time()

print(f'{"Ep":>4}  {"TrDice":>8}  {"VaDice":>8}  {"Best":>8}  {"Gap":>6}  {"Sec":>5}')
print('─'*52)

for ep in range(start_ep,EPOCHS+1):
    model.train();td_b=[];t0=time.time();optim.zero_grad(set_to_none=True)
    for imgs,msks in tr_dl:
        imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
        with autocast(): outs=model(imgs);loss=loss_fn(outs,msks)
        scaler.scale(loss).backward()
        scaler.unscale_(optim);nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optim);scaler.update();optim.zero_grad(set_to_none=True)
        with torch.no_grad(): _,d=get_dice(outs[0],msks);td_b.append(d)
    sched.step();td=float(np.mean(td_b));ep_s=time.time()-t0

    model.eval();Dc=defaultdict(list)
    with torch.no_grad():
        for imgs,msks in va_dl:
            imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
            with autocast(): out=model(imgs)
            D,_=get_dice(out,msks)
            for c,v in D.items(): Dc[c].extend(v)
    all_v=[v for vs in Dc.values() for v in vs]
    vd=float(np.mean(all_v)) if all_v else 0;gap=td-vd

    if vd>best:
        best=vd;no_imp=0
        pc={CN[c]:float(np.mean(v)) for c,v in Dc.items() if v}
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'best_dice':best,'per_class_dice':pc},CKPT_BEST)
        # Also save JSON summary
        with open(OUTPUT_DIR/'results.json','w') as f:
            json.dump({'epoch':ep,'best_dice':best,'per_class':pc},f,indent=2)
    else: no_imp+=1

    star='  ★' if vd==best else ''
    print(f'{ep:>4}  {td:>8.4f}  {vd:>8.4f}  {best:>8.4f}  {gap:>+6.3f}  {ep_s:>4.0f}s{star}')
    if vd>=0.90: print('\n🎯 TARGET ACHIEVED: Dice >= 0.90!'); break
    if vd>=0.85: print(f'  📈 Dice {vd:.4f} >= 0.85!')
    if no_imp>=50: print('\nEarly stop'); break

ttot=(time.time()-t0tot)/3600
print(f'\n✓ Training complete: {ttot:.2f}h | Best Dice: {best:.4f}')
print(f'✓ Checkpoint: {CKPT_BEST}')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 5: Final Results
# ═══════════════════════════════════════════════════════
import torch, numpy as np, json
from pathlib import Path

ckpt_path = Path('/kaggle/working/best_model.pth')
if ckpt_path.exists():
    ck = torch.load(str(ckpt_path), map_location='cpu')
    print('='*55)
    print(f'  FINAL RESULTS — ATM-Net++')
    print('='*55)
    print(f'  Best epoch : {ck["epoch"]}')
    print(f'  Best Dice  : {ck["best_dice"]:.4f}')
    pc = ck.get('per_class_dice', {})
    vert=[v for k,v in pc.items() if 'Vert' in k]
    ivd =[v for k,v in pc.items() if 'IVD'  in k]
    if vert: print(f'  Vertebrae  : {np.mean(vert):.4f}')
    if ivd:  print(f'  IVD        : {np.mean(ivd):.4f}')
    print(f'  Overall    : {np.mean(list(pc.values())):.4f}')
    print()
    print('  Per-class:')
    for name,d in sorted(pc.items(),key=lambda x:-x[1]):
        bar='█'*int(d*25); tag='★' if d>=0.90 else '✓' if d>=0.80 else '·' if d>=0.70 else ''
        print(f'    {name:<18} {d:.4f}  {bar} {tag}')
    print()
    print('Output files saved to /kaggle/working/:')
    import os
    for f in os.listdir('/kaggle/working/'):
        size = os.path.getsize(f'/kaggle/working/{f}') // 1024
        print(f'  {f}  ({size} KB)')
else:
    print('No checkpoint yet — run Step 4 first')